# Instacart — Data Quality Assessment

## Objective

This notebook assesses the quality of the imported Instacart tables before creating the `cleaned_data` layer.

The validation covers:

- table availability;
- column names and data types;
- missing values;
- duplicate keys;
- invalid categorical and numerical values;
- referential integrity across tables;
- order-sequence consistency;
- cart-position sequence consistency.

The source tables are stored in `workspace.imported_data`.

## 1. Setup and Data Loading

This section defines the source tables, verifies that they are available in Databricks, and loads them as Spark DataFrames for the quality checks that follow.

### 1.1 Configuration


In [0]:
from pyspark.sql import functions as F

In [0]:
catalog_name = "workspace"
imported_data_schema = "imported_data"

table_names = [
    "aisles",
    "departments",
    "products",
    "orders",
    "order_products_prior",
    "order_products_train"
]

print("Tables configured:", len(table_names))

Tables configured: 6


### 1.2 Verify Table Availability

Confirm that the expected imported tables exist in the `workspace.imported_data` schema.


In [0]:
available_tables_df = spark.sql(
    f"SHOW TABLES IN {catalog_name}.{imported_data_schema}"
)

display(available_tables_df)

database,tableName,isTemporary
imported_data,aisles,false
imported_data,departments,false
imported_data,order_products_prior,false
imported_data,order_products_train,false
imported_data,orders,false
imported_data,products,false


### 1.3 Load Imported Tables

Load the six imported Delta tables into a dictionary of Spark DataFrames so they can be reused throughout the notebook.


In [0]:

imported_data_dfs = {}

for table_name in table_names:
    
    full_table_name = (
        f"{catalog_name}.{imported_data_schema}.{table_name}"
    )
    
    imported_data_dfs[table_name] = spark.table(full_table_name)
    
    print(f"Loaded: {full_table_name}")

Loaded: workspace.imported_data.aisles
Loaded: workspace.imported_data.departments
Loaded: workspace.imported_data.products
Loaded: workspace.imported_data.orders
Loaded: workspace.imported_data.order_products_prior
Loaded: workspace.imported_data.order_products_train


## 2. Schema Validation

Inspect every column, data type, and nullability setting to confirm that the imported tables match the expected structure.


In [0]:

schema_inventory = []

for table_name, df in imported_data_dfs.items():
    
    for field in df.schema.fields:
        schema_inventory.append(
            (
                table_name,
                field.name,
                field.dataType.simpleString(),
                field.nullable
            )
        )

schema_inventory_df = spark.createDataFrame(
    schema_inventory,
    [
        "table_name",
        "column_name",
        "data_type",
        "nullable"
    ]
)

display(
    schema_inventory_df.orderBy(
        "table_name",
        "column_name"
    )
)

table_name,column_name,data_type,nullable
aisles,_ingested_at,timestamp,true
aisles,_source_file,string,true
aisles,aisle,string,true
aisles,aisle_id,int,true
departments,_ingested_at,timestamp,true
departments,_source_file,string,true
departments,department,string,true
departments,department_id,int,true
order_products_prior,_ingested_at,timestamp,true
order_products_prior,_source_file,string,true


## 3. Missing-Value Analysis

Missing values are profiled separately for reference tables and transactional tables.

The reusable profiling function calculates:

- total row count;
- null count by column;
- null percentage by column.


### 3.1 Reusable Null-Profiling Function


In [0]:


def create_null_profile(df, table_name):
    """
    Calculate row count and null statistics for every column
    using one Spark aggregation.
    """

    expressions = [
        F.count(F.lit(1)).alias("_total_rows")
    ]

    expressions += [
        F.sum(
            F.when(F.col(column).isNull(), 1).otherwise(0)
        ).alias(column)
        for column in df.columns
    ]

    result = df.agg(*expressions).first().asDict()

    total_rows = result["_total_rows"]

    profile_rows = []

    for column in df.columns:
        null_count = int(result[column] or 0)

        null_percentage = (
            round((null_count / total_rows) * 100, 4)
            if total_rows > 0
            else 0.0
        )

        profile_rows.append(
            (
                table_name,
                column,
                total_rows,
                null_count,
                null_percentage
            )
        )

    return profile_rows

### 3.2 Reference Tables

Profile null values in `departments`, `aisles`, and `products`.


In [0]:
reference_tables = [
    "departments",
    "aisles",
    "products"
]

null_profile_rows = []

for table_name in reference_tables:
    
    table_profile = create_null_profile(
        imported_data_dfs[table_name],
        table_name
    )
    
    null_profile_rows.extend(table_profile)

reference_null_profile_df = spark.createDataFrame(
    null_profile_rows,
    """
        table_name STRING,
        column_name STRING,
        row_count LONG,
        null_count LONG,
        null_percentage DOUBLE
    """
)

display(
    reference_null_profile_df.orderBy(
        "table_name",
        F.desc("null_percentage")
    )
)

table_name,column_name,row_count,null_count,null_percentage
aisles,aisle_id,134,0,0.0
aisles,aisle,134,0,0.0
aisles,_source_file,134,0,0.0
aisles,_ingested_at,134,0,0.0
departments,department_id,21,0,0.0
departments,department,21,0,0.0
departments,_source_file,21,0,0.0
departments,_ingested_at,21,0,0.0
products,aisle_id,49688,1,0.002
products,department_id,49688,1,0.002


## 4. Duplicate Primary Key Check

This check verifies that the primary key columns in the reference tables contain no duplicate values.

Each key should uniquely identify one record:

- `department_id` in `departments`
- `aisle_id` in `aisles`
- `product_id` in `products`

Any duplicate key would indicate a data-quality issue that should be investigated before proceeding to the next processing layer.

In [0]:
primary_keys = {
    "departments": ["department_id"],
    "aisles": ["aisle_id"],
    "products": ["product_id"]
}

duplicate_key_results = []

for table_name, key_columns in primary_keys.items():
    
    df = imported_data_dfs[table_name]
    
    duplicate_groups = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    
    duplicate_key_results.append(
        (
            table_name,
            ", ".join(key_columns),
            duplicate_groups
        )
    )

duplicate_keys_df = spark.createDataFrame(
    duplicate_key_results,
    [
        "table_name",
        "key_columns",
        "duplicate_key_groups"
    ]
)

display(duplicate_keys_df)

table_name,key_columns,duplicate_key_groups
departments,department_id,0
aisles,aisle_id,0
products,product_id,0


## 5. Missing Product References

This check identifies products with missing classification references.

Each product is expected to have:

- an `aisle_id`
- a `department_id`

Products with a missing value in either of these columns may not be correctly linked to their corresponding aisle or department and should be reviewed before further processing.

In [0]:
products_df = imported_data_dfs["products"]

products_with_missing_references_df = (
    products_df
    .filter(
        F.col("aisle_id").isNull()
        | F.col("department_id").isNull()
    )
)

display(products_with_missing_references_df)

product_id,product_name,aisle_id,department_id,_source_file,_ingested_at
6816,"""Scotch Kids 5"""" Scissors",null,null,products.csv,2026-07-26T23:04:55.994Z


## 6. Validate Product References

This check verifies that the `aisle_id` and `department_id` values stored in the `products` table actually exist in their corresponding reference tables.

The validation checks for:

- products linked to a non-existent `aisle_id`
- products linked to a non-existent `department_id`

A `left_anti` join is used to identify product records whose reference IDs cannot be found in the `aisles` or `departments` tables.

In [0]:

from pyspark.sql import functions as F

products_df = spark.table("workspace.imported_data.products")
aisles_df = spark.table("workspace.imported_data.aisles")
departments_df = spark.table("workspace.imported_data.departments")

# Products whose aisle_id does not match any existing aisle
invalid_aisle_references_df = (
    products_df
    .filter(F.col("aisle_id").isNotNull())
    .join(
        aisles_df.select("aisle_id"),
        on="aisle_id",
        how="left_anti"
    )
)

# Products whose department_id does not match any existing department
invalid_department_references_df = (
    products_df
    .filter(F.col("department_id").isNotNull())
    .join(
        departments_df.select("department_id"),
        on="department_id",
        how="left_anti"
    )
)

print(
    "Produits avec un aisle_id inexistant :",
    invalid_aisle_references_df.count()
)

print(
    "Produits avec un department_id inexistant :",
    invalid_department_references_df.count()
)

Produits avec un aisle_id inexistant : 0
Produits avec un department_id inexistant : 0


## 7. Null Value Profiling in Transactional Tables

This analysis measures the number and percentage of missing values in the main transactional tables.

The following tables are evaluated:

- `orders`
- `order_products_prior`
- `order_products_train`

For each column, the analysis calculates:

- the total number of rows
- the number of null values
- the percentage of null values

This helps identify missing-data patterns that may require further validation before creating the cleaned data layer.

In [0]:
from pyspark.sql import functions as F

orders_df = spark.table("workspace.imported_data.orders")

order_products_prior_df = spark.table(
    "workspace.imported_data.order_products_prior"
)

order_products_train_df = spark.table(
    "workspace.imported_data.order_products_train"
)


def create_null_profile(df, table_name):
    """
    Calculate the number and percentage of null values
    for each column in a table.
    """

    aggregation_expressions = [
        F.count(F.lit(1)).alias("_total_rows")
    ]

    aggregation_expressions += [
        F.sum(
            F.when(F.col(column).isNull(), 1).otherwise(0)
        ).alias(column)
        for column in df.columns
    ]

    result = df.agg(*aggregation_expressions).first().asDict()

    total_rows = result["_total_rows"]
    profile_rows = []

    for column in df.columns:
        null_count = int(result[column] or 0)

        null_percentage = (
            round(null_count / total_rows * 100, 4)
            if total_rows > 0
            else 0.0
        )

        profile_rows.append(
            (
                table_name,
                column,
                total_rows,
                null_count,
                null_percentage
            )
        )

    return profile_rows


transaction_tables = {
    "orders": orders_df,
    "order_products_prior": order_products_prior_df,
    "order_products_train": order_products_train_df
}

transaction_null_profile_rows = []

for table_name, df in transaction_tables.items():

    print(f"Profilage de la table {table_name}...")

    transaction_null_profile_rows.extend(
        create_null_profile(df, table_name)
    )


transaction_null_profile_df = spark.createDataFrame(
    transaction_null_profile_rows,
    """
    table_name STRING,
    column_name STRING,
    row_count LONG,
    null_count LONG,
    null_percentage DOUBLE
    """
)

display(
    transaction_null_profile_df.orderBy(
        "table_name",
        F.desc("null_percentage")
    )
)

Profilage de la table orders...
Profilage de la table order_products_prior...
Profilage de la table order_products_train...


table_name,column_name,row_count,null_count,null_percentage
order_products_prior,order_id,32434489,0,0.0
order_products_prior,product_id,32434489,0,0.0
order_products_prior,add_to_cart_order,32434489,0,0.0
order_products_prior,reordered,32434489,0,0.0
order_products_prior,_source_file,32434489,0,0.0
order_products_prior,_ingested_at,32434489,0,0.0
order_products_train,order_id,1384617,0,0.0
order_products_train,product_id,1384617,0,0.0
order_products_train,add_to_cart_order,1384617,0,0.0
order_products_train,reordered,1384617,0,0.0


## 8. Validate `days_since_prior_order` Null Values

This check verifies whether the null values found in `days_since_prior_order` are legitimate.

A null value is expected when `order_number = 1`, because the customer's first order has no previous order to compare with.

The validation checks:

- first orders with a null `days_since_prior_order` → expected
- first orders with a non-null value → potentially inconsistent
- later orders with a null `days_since_prior_order` → potentially inconsistent

This helps confirm that the missing values follow the expected business logic of the dataset.

In [0]:
from pyspark.sql import functions as F

orders_df = spark.table("workspace.bronze.orders")

days_since_prior_validation_df = (
    orders_df
    .agg(
        F.sum(
            F.when(
                (F.col("order_number") == 1)
                & F.col("days_since_prior_order").isNull(),
                1
            ).otherwise(0)
        ).alias("premieres_commandes_avec_null"),

        F.sum(
            F.when(
                (F.col("order_number") == 1)
                & F.col("days_since_prior_order").isNotNull(),
                1
            ).otherwise(0)
        ).alias("premieres_commandes_avec_valeur"),

        F.sum(
            F.when(
                (F.col("order_number") > 1)
                & F.col("days_since_prior_order").isNull(),
                1
            ).otherwise(0)
        ).alias("commandes_suivantes_avec_null")
    )
)

display(days_since_prior_validation_df)

premieres_commandes_avec_null,premieres_commandes_avec_valeur,commandes_suivantes_avec_null
206209,0,0


## 9. Duplicate Key Check in Transactional Tables

This check verifies that the key columns expected to be unique in the transactional tables do not contain duplicate records.

The expected unique keys are:

- `order_id` in `orders`
- the combination of `order_id` and `product_id` in `order_products_prior`
- the combination of `order_id` and `product_id` in `order_products_train`

Any duplicate key group may indicate duplicated transactional records and should be reviewed before further processing.

In [0]:
from pyspark.sql import functions as F

orders_df = spark.table("workspace.imported_data.orders")

order_products_prior_df = spark.table(
    "workspace.imported_data.order_products_prior"
)

order_products_train_df = spark.table(
    "workspace.imported_data.order_products_train"
)

tables_and_keys = {
    "orders": (
        orders_df,
        ["order_id"]
    ),
    "order_products_prior": (
        order_products_prior_df,
        ["order_id", "product_id"]
    ),
    "order_products_train": (
        order_products_train_df,
        ["order_id", "product_id"]
    )
}

duplicate_results = []

for table_name, (df, key_columns) in tables_and_keys.items():

    print(f"Vérification des doublons dans {table_name}...")

    duplicate_groups = (
        df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    duplicate_results.append(
        (
            table_name,
            ", ".join(key_columns),
            duplicate_groups
        )
    )

duplicate_results_df = spark.createDataFrame(
    duplicate_results,
    [
        "table_name",
        "key_columns",
        "duplicate_key_groups"
    ]
)

display(duplicate_results_df)

Vérification des doublons dans orders...
Vérification des doublons dans order_products_prior...
Vérification des doublons dans order_products_train...


table_name,key_columns,duplicate_key_groups
orders,order_id,0
order_products_prior,"order_id, product_id",0
order_products_train,"order_id, product_id",0


## 10. Categorical and Numerical Value Validation

This check identifies invalid categorical and numerical values in the `orders` table.

The validation verifies that:

- `eval_set` contains only `prior`, `train`, or `test`
- `order_dow` is between 0 and 6
- `order_hour_of_day` is between 0 and 23
- `order_number` is greater than or equal to 1
- `days_since_prior_order` is not negative

Any value outside these expected ranges may indicate a data-quality issue that should be reviewed before further processing.

In [0]:

from pyspark.sql import functions as F

orders_df = spark.table("workspace.imported_data.orders")

order_products_prior_df = spark.table(
    "workspace.imported_data.order_products_prior"
)

order_products_train_df = spark.table(
    "workspace.imported_data.order_products_train"
)

orders_quality_checks_df = (
    orders_df
    .agg(
        F.sum(
            F.when(
                ~F.col("eval_set").isin("prior", "train", "test"),
                1
            ).otherwise(0)
        ).alias("eval_set_invalide"),

        F.sum(
            F.when(
                ~F.col("order_dow").between(0, 6),
                1
            ).otherwise(0)
        ).alias("jour_semaine_invalide"),

        F.sum(
            F.when(
                ~F.col("order_hour_of_day").between(0, 23),
                1
            ).otherwise(0)
        ).alias("heure_commande_invalide"),

        F.sum(
            F.when(
                F.col("order_number") < 1,
                1
            ).otherwise(0)
        ).alias("numero_commande_invalide"),

        F.sum(
            F.when(
                F.col("days_since_prior_order") < 0,
                1
            ).otherwise(0)
        ).alias("delai_negatif")
    )
)

display(orders_quality_checks_df)

eval_set_invalide,jour_semaine_invalide,heure_commande_invalide,numero_commande_invalide,delai_negatif
0,0,0,0,0


### 10.1 Validate Order-Product Values

This check validates the numerical and categorical values in the `order_products_prior` and `order_products_train` tables.

The validation verifies that:

- `add_to_cart_order` is greater than or equal to 1
- `reordered` contains only the values 0 or 1

Any value outside these expected rules may indicate a data-quality issue that should be reviewed before further processing.

In [0]:
order_product_quality_results = []

order_product_tables = {
    "order_products_prior": order_products_prior_df,
    "order_products_train": order_products_train_df
}

for table_name, df in order_product_tables.items():

    result = (
        df.agg(
            F.sum(
                F.when(
                    F.col("add_to_cart_order") < 1,
                    1
                ).otherwise(0)
            ).alias("position_panier_invalide"),

            F.sum(
                F.when(
                    ~F.col("reordered").isin(0, 1),
                    1
                ).otherwise(0)
            ).alias("valeur_reordered_invalide")
        )
        .first()
    )

    order_product_quality_results.append(
        (
            table_name,
            int(result["position_panier_invalide"] or 0),
            int(result["valeur_reordered_invalide"] or 0)
        )
    )

order_product_quality_df = spark.createDataFrame(
    order_product_quality_results,
    [
        "table_name",
        "position_panier_invalide",
        "valeur_reordered_invalide"
    ]
)

display(order_product_quality_df)

table_name,position_panier_invalide,valeur_reordered_invalide
order_products_prior,0,0
order_products_train,0,0


## 11. Cross-Table Referential Integrity

This check verifies that the relationships between the transactional and reference tables are consistent.

The validation checks whether:

- every `order_id` in `order_products_prior` exists in `orders`
- every `order_id` in `order_products_train` exists in `orders`
- every `product_id` in the order-product tables exists in `products`
- orders stored in `order_products_prior` have `eval_set = "prior"`
- orders stored in `order_products_train` have `eval_set = "train"`

Any failed relationship may indicate a referential-integrity issue between the tables.

In [0]:
from pyspark.sql import functions as F

orders_keys_df = (
    spark.table("workspace.imported_data.orders")
    .select("order_id", "eval_set")
)

products_keys_df = (
    spark.table("workspace.imported_data.products")
    .select("product_id")
)

prior_df = spark.table(
    "workspace.imported_data.order_products_prior"
)

train_df = spark.table(
    "workspace.imported_data.order_products_train"
)

# 1. Check for order_id values that do not exist in orders
prior_orders_missing = (
    prior_df
    .select("order_id")
    .distinct()
    .join(
        orders_keys_df.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .count()
)

train_orders_missing = (
    train_df
    .select("order_id")
    .distinct()
    .join(
        orders_keys_df.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .count()
)

# 2. Check for product_id values that do not exist in products
prior_products_missing = (
    prior_df
    .select("product_id")
    .distinct()
    .join(
        products_keys_df,
        on="product_id",
        how="left_anti"
    )
    .count()
)

train_products_missing = (
    train_df
    .select("product_id")
    .distinct()
    .join(
        products_keys_df,
        on="product_id",
        how="left_anti"
    )
    .count()
)

# 3. Check consistency between each table and eval_set
prior_wrong_eval_set = (
    prior_df
    .select("order_id")
    .distinct()
    .join(
        orders_keys_df,
        on="order_id",
        how="inner"
    )
    .filter(F.col("eval_set") != "prior")
    .count()
)

train_wrong_eval_set = (
    train_df
    .select("order_id")
    .distinct()
    .join(
        orders_keys_df,
        on="order_id",
        how="inner"
    )
    .filter(F.col("eval_set") != "train")
    .count()
)

integrity_results = [
    ("prior_order_id_inexistant", prior_orders_missing),
    ("train_order_id_inexistant", train_orders_missing),
    ("prior_product_id_inexistant", prior_products_missing),
    ("train_product_id_inexistant", train_products_missing),
    ("prior_eval_set_incoherent", prior_wrong_eval_set),
    ("train_eval_set_incoherent", train_wrong_eval_set)
]

integrity_results_df = spark.createDataFrame(
    integrity_results,
    ["controle", "nombre_anomalies"]
)

display(integrity_results_df)

controle,nombre_anomalies
prior_order_id_inexistant,0
train_order_id_inexistant,0
prior_product_id_inexistant,0
train_product_id_inexistant,0
prior_eval_set_incoherent,0
train_eval_set_incoherent,0


## 12. Order Sequence Consistency Check

This check verifies that each customer's order history follows a consistent chronological sequence.

For every `user_id`, the validation checks that:

- the first `order_number` starts at 1
- order numbers form a continuous sequence without gaps or duplicates
- each customer has exactly one final order in either the `train` or `test` set
- the final `train` or `test` order corresponds to the customer's highest `order_number`

Any inconsistency may indicate an issue in the customer order sequence.

In [0]:
from pyspark.sql import functions as F

orders_df = spark.table("workspace.imported_data.orders")

# Summarize orders for each customer
user_order_summary_df = (
    orders_df
    .groupBy("user_id")
    .agg(
        F.min("order_number").alias("min_order_number"),
        F.max("order_number").alias("max_order_number"),
        F.count("*").alias("nombre_lignes"),
        F.countDistinct("order_number").alias(
            "nombre_numeros_distincts"
        ),

        F.sum(
            F.when(
                F.col("eval_set").isin("train", "test"),
                1
            ).otherwise(0)
        ).alias("nombre_commandes_finales"),

        F.max(
            F.when(
                F.col("eval_set").isin("train", "test"),
                F.col("order_number")
            )
        ).alias("numero_commande_finale")
    )
)

# Perform global consistency checks
order_sequence_checks_df = (
    user_order_summary_df
    .agg(
        F.sum(
            F.when(
                F.col("min_order_number") != 1,
                1
            ).otherwise(0)
        ).alias("utilisateurs_sans_commande_1"),

        F.sum(
            F.when(
                (
                    F.col("nombre_lignes")
                    != F.col("max_order_number")
                )
                |
                (
                    F.col("nombre_numeros_distincts")
                    != F.col("max_order_number")
                ),
                1
            ).otherwise(0)
        ).alias("utilisateurs_avec_sequence_incoherente"),

        F.sum(
            F.when(
                F.col("nombre_commandes_finales") != 1,
                1
            ).otherwise(0)
        ).alias("utilisateurs_sans_unique_train_test"),

        F.sum(
            F.when(
                F.col("numero_commande_finale")
                != F.col("max_order_number"),
                1
            ).otherwise(0)
        ).alias("utilisateurs_dont_finale_pas_derniere")
    )
)

display(order_sequence_checks_df)

utilisateurs_sans_commande_1,utilisateurs_avec_sequence_incoherente,utilisateurs_sans_unique_train_test,utilisateurs_dont_finale_pas_derniere
0,0,0,0


## 13. Cart Position Sequence Consistency Check

This check verifies that product positions within each order form a valid and continuous sequence.

For every `order_id`, the validation checks that:

- `add_to_cart_order` starts at 1
- the maximum cart position matches the total number of products in the order
- cart positions are unique, with no duplicated positions
- the sequence contains no gaps

Any inconsistency may indicate an issue in the ordering of products within a customer's cart.

In [0]:
from pyspark.sql import functions as F

prior_df = spark.table(
    "workspace.imported_data.order_products_prior"
)

train_df = spark.table(
    "workspace.imported_data.order_products_train"
)

cart_sequence_results = []

tables_to_check = {
    "order_products_prior": prior_df,
    "order_products_train": train_df
}

for table_name, df in tables_to_check.items():
# Summarize cart positions for each order
    order_cart_summary_df = (
        df
        .groupBy("order_id")
        .agg(
            F.min("add_to_cart_order").alias("position_min"),
            F.max("add_to_cart_order").alias("position_max"),
            F.count("*").alias("nombre_produits"),
            F.countDistinct("add_to_cart_order").alias(
                "nombre_positions_distinctes"
            )
        )
    )
    # Identify orders with inconsistent cart-position sequences
    invalid_orders = (
        order_cart_summary_df
        .filter(
            (F.col("position_min") != 1)
            |
            (
                F.col("position_max")
                != F.col("nombre_produits")
            )
            |
            (
                F.col("nombre_positions_distinctes")
                != F.col("nombre_produits")
            )
        )
        .count()
    )

    cart_sequence_results.append(
        (table_name, invalid_orders)
    )

cart_sequence_results_df = spark.createDataFrame(
    cart_sequence_results,
    [
        "table_name",
        "commandes_avec_sequence_panier_incoherente"
    ]
)

display(cart_sequence_results_df)

table_name,commandes_avec_sequence_panier_incoherente
order_products_prior,0
order_products_train,0


## 14. Conclusion

The data quality checks show that the Instacart dataset is mostly clean and ready for the next step.

The main results are:

- No duplicate IDs were found in the `departments`, `aisles`, and `products` tables.
- One product has missing `aisle_id` and `department_id` values.
- All other product references match existing aisles and departments.
- The null values in `days_since_prior_order` are normal because they only appear for customers' first orders.
- No duplicate records were found in the main transaction tables.
- No invalid values were found in the columns that were checked.
- The links between orders, products, and the other tables are correct.
- The `eval_set` values are correct for the `prior` and `train` tables.
- Customer order numbers follow the correct sequence.
- Product positions inside each order also follow the correct sequence.

Overall, the dataset has good data quality. The only problem found is one product with missing aisle and department information.

The next step is to create the `cleaned_data` layer and handle this missing information before continuing with the analysis.